In [2]:
%pip install --upgrade openai pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.7 MB/s eta 0:00:00
  Attempting uninstall: h11
    Found existing installation: h11 0.14.0
    Uninstalling h11-0.14.0:
      Successfully uninstalled h11-0.14.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.1
    Uninstalling pandas-2.3.1:
      Successfully uninstalled pandas-2.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.16.1 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.10.3 which is incompatible.
ydata-profiling 4.16.1 requires numpy<2.2,>=1.16.0, but you have numpy 2.2.6 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import pandas as pd
from openai import OpenAI

In [2]:
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [3]:
import pandas as pd

# Paths (change if your filenames differ)
questions_path = "QACP_Q.json"
answers_path = "QACP_A.csv"

# Read JSON list of objects -> DataFrame
q_df = pd.read_json(questions_path)    # contains id, Question, etc.

# Read CSV with answers
a_df = pd.read_csv(answers_path, encoding="gbk")    # contains id, Answer, etc.

print("Questions columns:", q_df.columns.tolist())
print("Answers columns:", a_df.columns.tolist())

q_df.head(), a_df.head()

Questions columns: ['id', 'Knowledge Point', 'Question Type', 'Answer Type', 'Question']
Answers columns: ['id', 'Answer']


(   id         Knowledge Point              Question Type        Answer Type  \
 0   1  Introduction to Python  Experience in Programming  Explain in Detail   
 1   2  Introduction to Python                   Beginner  Explain in Detail   
 2   3  Introduction to Python                   Beginner  Explain in Detail   
 3   4  Introduction to Python                   Beginner  Explain in Detail   
 4   5  Introduction to Python                   Beginner  Explain in Detail   
 
                   Question  
 0               Python是什么？  
 1               Python是什么？  
 2          Python的发展历程是什么？  
 3          Python语言的缺点是什么？  
 4  为什么Python是初学者入门编程的理想选择？  ,
    id                                             Answer
 0   1  Python语言从1991年诞生至今已经30年了，近年来随着互联网的迅速发展以及人工智能、物...
 1   2  ## python是什么\nPython是一种非常流行的编程语言，就像中国人之间用汉语沟通一...
 2   3  Python，一种以简洁清晰的语法和强大功能著称的编程语言，其发展历程主要分为以下几个关键阶...
 3   4  Python，作为一种广泛使用的编程语言，不仅以其简洁清晰的语法和强大功能而闻名，还因为对初...
 4   5  对于初学者，我们可以从两方面判断：\n1.功能和效率：功能越强大、开发效率越高

In [4]:
# Merge questions and answers on "id"
df = q_df.merge(a_df, on="id", how="inner")

# Rename to simpler column names for the translation step
df = df.rename(columns={
    "Question": "question_zh",
    "Answer": "answer_zh",
})

# Quick check
df[["id", "question_zh", "answer_zh"]].head()

,id,question_zh,answer_zh
0,1,Python是什么？,Python语言从1991年诞生至今已经30年了，近年来随着互联网的迅速发展以及人工智能、物...
1,2,Python是什么？,## python是什么\nPython是一种非常流行的编程语言，就像中国人之间用汉语沟通一...
2,3,Python的发展历程是什么？,Python，一种以简洁清晰的语法和强大功能著称的编程语言，其发展历程主要分为以下几个关键阶...
3,4,Python语言的缺点是什么？,Python，作为一种广泛使用的编程语言，不仅以其简洁清晰的语法和强大功能而闻名，还因为对初...
4,5,为什么Python是初学者入门编程的理想选择？,对于初学者，我们可以从两方面判断：\n1.功能和效率：功能越强大、开发效率越高，应用前景越好...


In [5]:
SYSTEM_PROMPT = """
You are a professional translator for a Chinese single-turn Q&A dataset about Python programming.

Task:
- Translate all natural language from Simplified Chinese to Modern Standard Arabic.
- The dataset contains questions and answers for Python learners. Explanations should be clear, didactic, and in Modern Standard Arabic (no dialect).

VERY IMPORTANT CODE RULES:
- Do NOT translate or modify any Python code.
- Treat as code anything that is:
  - Inside triple backticks ``` ```
  - Inside inline backticks `like_this`
  - Inside <code>...</code> tags
  - Or clearly Python syntax (e.g., def, for, while, if, else, return, import, print, :, indentation blocks, list/dict literals, etc.).
- All such code must be preserved exactly: same words, same indentation, same spacing.

OTHER RULES:
- Preserve overall structure:
  - Keep headings, bullet points, and line breaks.
- If Chinese appears mixed with code comments (like `# 注释` inside code), DO NOT translate it. Keep the entire code line exactly as is.
- Do not add extra explanations. Just translate the Chinese text and keep all code unchanged.
"""

In [6]:
def translate_cn_to_ar(text_cn: str, model: str = "gpt-5-nano") -> str:
    if not isinstance(text_cn, str) or text_cn.strip() == "":
        return ""

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text_cn},
        ],
    )

    return resp.choices[0].message.content.strip()

In [7]:
df.head()
df.columns

Index(['id', 'Knowledge Point', 'Question Type', 'Answer Type', 'question_zh',
       'answer_zh'],
      dtype='object')

In [8]:
sample_row = df.iloc[0]  # you can change 0 to any index

print("=== Chinese Question ===")
print(sample_row["question_zh"])

print("\n=== Chinese Answer (first 300 chars) ===")
print(sample_row["answer_zh"][:300])

print("\n=== Arabic Question (translated) ===")
print(translate_cn_to_ar(sample_row["question_zh"])[:500])

print("\n=== Arabic Answer (translated, first 500 chars) ===")
print(translate_cn_to_ar(sample_row["answer_zh"])[:500])

=== Chinese Question ===
Python是什么？

=== Chinese Answer (first 300 chars) ===
Python语言从1991年诞生至今已经30年了，近年来随着互联网的迅速发展以及人工智能、物联网、数据分析等热门行业凸起，Python作为其重要的支持语言之一越来越流行，热度也不断上升。Python的应用领域十分广泛，不论是运维、架构、web开发、数据分析还是验证算法它都有完善的框架能够使用，可以说，在爬虫，机器学习，人工智能这些方面python基本上是一家独大了。
- 简短易读
与其他编程语言相比，比如Java或C++，Python的代码通常更简短、更易读。例如，你可以用几行Python代码完成的任务，在Java或C++中可能需要更多的代码行。
- 开发效率更高
python是一种解释型语言

=== Arabic Question (translated) ===
ما هو بايثون؟

=== Arabic Answer (translated, first 500 chars) ===
- قصير وسهل القراءة
- مقارنةً مع لغات برمجة أخرى، مثل Java أو C++، عادةً ما تكون شفرة بايثون أقصر وأكثر قابلية للقراءة. على سبيل المثال، المهمة التي يمكنك إكمالها ببضع أسطر من بايثون قد تتطلب مزيدًا من الأسطر في Java أو C++.
- كفاءة تطوير أعلى
- بايثون هي لغة تفسيرية، يمكنك تشغيل الشفرة المكتوبة مباشرة، دون الحاجة إلى ترجمتها مثل C++ أو Java قبل ذلك. هذا يجعل عملية البرمجة والاختبار أسرع وأكثر مرونة.
- قوة وظيفية وتطبيقات واسعة
- بايثون قوي وظيفيًا، يمكنك استخدامها لبناء مواقع الويب والتطبيقات، 


In [9]:
df["question_ar"] = ""
df["answer_ar"] = ""

for idx, row in df.iterrows():
    q_cn = row["question_zh"]
    a_cn = row["answer_zh"]

    df.at[idx, "question_ar"] = translate_cn_to_ar(q_cn)
    df.at[idx, "answer_ar"] = translate_cn_to_ar(a_cn)

    if (idx + 1) % 20 == 0:
        print(f"Translated {idx + 1} rows...")

Translated 20 rows...
Translated 40 rows...
Translated 60 rows...
Translated 80 rows...
Translated 100 rows...
Translated 120 rows...
Translated 140 rows...
Translated 160 rows...
Translated 180 rows...
Translated 200 rows...
Translated 220 rows...
Translated 240 rows...
Translated 260 rows...
Translated 280 rows...
Translated 300 rows...
Translated 320 rows...
Translated 340 rows...
Translated 360 rows...
Translated 380 rows...
Translated 400 rows...
Translated 420 rows...
Translated 440 rows...
Translated 460 rows...
Translated 480 rows...
Translated 500 rows...
Translated 520 rows...


In [10]:
output_path = "QACP_Python_QA_zh_ar.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("Saved to:", output_path)

Saved to: QACP_Python_QA_zh_ar.csv
